In [1]:
from google.colab import files
uploaded = files.upload()

Saving DataCoSupplyChainDataset.csv to DataCoSupplyChainDataset.csv


In [2]:
import pandas as pd

df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')

print(df.shape)        # (rows, columns) — how big is this dataset?
print(df.columns.tolist())  # what columns do we actually have?
df.head()               # first 5 rows — what does the data look like?

(180519, 53)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping d

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

Product Description    180519
Order Zipcode          155679
Customer Lname              8
Customer Zipcode            3
dtype: int64


In [4]:
print(df.duplicated().sum())

0


In [5]:
# Drop the columns we don't need
df = df.drop(columns=['Product Description', 'Order Zipcode', 'Customer Password', 'Customer Email', 'Product Image'])

# Drop the handful of rows missing Lname/Zipcode — negligible loss out of 180K rows
df = df.dropna(subset=['Customer Lname', 'Customer Zipcode'])

# Confirm we're clean
print(df.isnull().sum().sum())   # should print 0
print(df.shape)                   # how many rows/columns did we end up with?

0
(180508, 48)


In [6]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

print(df.dtypes[['order date (DateOrders)', 'shipping date (DateOrders)']])

order date (DateOrders)       datetime64[ns]
shipping date (DateOrders)    datetime64[ns]
dtype: object


In [7]:
df.to_csv('cleaned_supply_chain.csv', index=False)

from google.colab import files
files.download('cleaned_supply_chain.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import sqlite3

conn = sqlite3.connect('supply_chain.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

# sanity check — run a trivial query to confirm it worked
test = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM orders", conn)
print(test)

   total_rows
0      180508


In [11]:
query = """
SELECT
  "Order Region",
  COUNT(*) as total_orders,
  SUM(Late_delivery_risk) as late_orders,
  ROUND(100.0 * (1 - AVG(Late_delivery_risk)), 2) as on_time_pct
FROM orders
GROUP BY "Order Region"
ORDER BY on_time_pct ASC;
"""

In [12]:
result = pd.read_sql_query(query, conn)
print(result)

       Order Region  total_orders  late_orders  on_time_pct
0    Central Africa          1677          972        42.04
1        South Asia          7730         4350        43.73
2       East Africa          1852         1036        44.06
3    Western Europe         27107        15139        44.15
4    South of  USA           4045         2256        44.23
5       East of USA          6915         3849        44.34
6    Eastern Europe          3920         2182        44.34
7    Southeast Asia          9535         5295        44.47
8      Central Asia           553          306        44.67
9         West Asia          6009         3322        44.72
10       US Center           5887         3252        44.76
11  Central America         28341        15518        45.25
12     North Africa          3232         1762        45.48
13  Southern Europe          9430         5128        45.62
14     Eastern Asia          7279         3954        45.68
15    South America         14935       

In [13]:
query = """
SELECT
  "Order Region",
  "Category Name",
  ROUND(SUM(Sales), 2) as total_sales,
  RANK() OVER (PARTITION BY "Order Region" ORDER BY SUM(Sales) DESC) as sales_rank
FROM orders
GROUP BY "Order Region", "Category Name"
ORDER BY "Order Region", sales_rank
LIMIT 30;
"""

result = pd.read_sql_query(query, conn)
print(result)

   Order Region         Category Name  total_sales  sales_rank
0        Canada               Fishing     27598.62           1
1        Canada                Cleats     25675.72           2
2        Canada      Cardio Equipment     24587.55           3
3        Canada      Camping & Hiking     24298.38           4
4        Canada          Water Sports     18399.08           5
5        Canada       Women's Apparel     17500.00           6
6        Canada  Indoor/Outdoor Games     16693.32           7
7        Canada        Men's Footwear     15728.79           8
8        Canada         Shop By Sport      7448.22           9
9        Canada           Electronics      2451.37          10
10       Canada        Girls' Apparel      1329.90          11
11       Canada           Accessories      1149.54          12
12       Canada          Boxing & MMA       769.58          13
13       Canada            Golf Shoes       650.00          14
14       Canada            Golf Balls       573.65     

In [14]:
query = """
WITH delivery_stats AS (
  SELECT
    "Order Region",
    ROUND(100.0 * (1 - AVG(Late_delivery_risk)), 2) as on_time_pct,
    ROUND(SUM(Sales), 2) as total_sales
  FROM orders
  GROUP BY "Order Region"
)
SELECT *
FROM delivery_stats
ORDER BY total_sales DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
print(result)

      Order Region  on_time_pct  total_sales
0   Western Europe        44.15   5893666.41
1  Central America        45.25   5665712.10
2    South America        45.69   2960881.41
3  Northern Europe        45.94   2155012.07
4  Southern Europe        45.62   2047665.94
5          Oceania        45.98   2016654.20
6   Southeast Asia        44.47   1931525.80
7        Caribbean        46.92   1651019.33
8     West of USA         46.04   1571415.96
9       South Asia        43.73   1553465.10


In [15]:
# quick way to save your queries as a file for the repo
queries_text = """
-- Query 1: On-time delivery rate by region
SELECT "Order Region", COUNT(*) as total_orders, SUM(Late_delivery_risk) as late_orders,
ROUND(100.0 * (1 - AVG(Late_delivery_risk)), 2) as on_time_pct
FROM orders GROUP BY "Order Region" ORDER BY on_time_pct ASC;

-- Query 2: Top categories by sales, ranked within each region
SELECT "Order Region", "Category Name", ROUND(SUM(Sales), 2) as total_sales,
RANK() OVER (PARTITION BY "Order Region" ORDER BY SUM(Sales) DESC) as sales_rank
FROM orders GROUP BY "Order Region", "Category Name" ORDER BY "Order Region", sales_rank;

-- Query 3: Regions with high sales but poor on-time delivery (CTE)
WITH delivery_stats AS (
  SELECT "Order Region", ROUND(100.0 * (1 - AVG(Late_delivery_risk)), 2) as on_time_pct,
  ROUND(SUM(Sales), 2) as total_sales FROM orders GROUP BY "Order Region"
)
SELECT * FROM delivery_stats ORDER BY total_sales DESC LIMIT 10;
"""

with open('queries.sql', 'w') as f:
    f.write(queries_text)

files.download('queries.sql')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import matplotlib.pyplot as plt

correlation = df['Order Item Discount Rate'].corr(df['Late_delivery_risk'])
print(f"Correlation between discount rate and late delivery: {correlation:.4f}")

Correlation between discount rate and late delivery: 0.0004


In [17]:
shipping_analysis = df.groupby('Shipping Mode')['Late_delivery_risk'].agg(['mean', 'count'])
shipping_analysis['late_pct'] = (shipping_analysis['mean'] * 100).round(2)
print(shipping_analysis)

                    mean   count  late_pct
Shipping Mode                             
First Class     0.953222   27812     95.32
Same Day        0.457430    9737     45.74
Second Class    0.766343   35214     76.63
Standard Class  0.380723  107745     38.07


In [18]:
scheduled_by_mode = df.groupby('Shipping Mode')['Days for shipment (scheduled)'].mean().round(2)
print(scheduled_by_mode)

Shipping Mode
First Class       1.0
Same Day          0.0
Second Class      2.0
Standard Class    4.0
Name: Days for shipment (scheduled), dtype: float64


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Features we'll use — chosen because we've already seen they matter, or plausibly would
features = df[['Days for shipment (scheduled)', 'Order Item Discount Rate',
               'Order Item Quantity', 'Shipping Mode', 'Order Region']].copy()
target = df['Late_delivery_risk']

# Encode categorical columns to numbers
for col in ['Shipping Mode', 'Order Region']:
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col])

# Split into train/test — 80% to train the model, 20% held back to honestly evaluate it
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(144406, 5) (36102, 5)


In [20]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

model = DecisionTreeClassifier(max_depth=6, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("Precision:", round(precision_score(y_test, predictions), 4))
print("Recall:", round(recall_score(y_test, predictions), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.6974
Precision: 0.8444
Recall: 0.546

Confusion matrix:
[[14423  1982]
 [ 8943 10754]]


In [21]:
importance = pd.DataFrame({
    'feature': features.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importance)

                         feature  importance
3                  Shipping Mode    0.794905
0  Days for shipment (scheduled)    0.196856
4                   Order Region    0.004492
1       Order Item Discount Rate    0.002235
2            Order Item Quantity    0.001512


In [22]:
df.to_csv('supply_chain_for_tableau.csv', index=False)
from google.colab import files
files.download('supply_chain_for_tableau.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# check the actual date range and count of orders in the last couple months
print(df['order date (DateOrders)'].max())
print(df['order date (DateOrders)'].min())

recent = df[df['order date (DateOrders)'] >= '2017-12-01']
print(recent.groupby(recent['order date (DateOrders)'].dt.to_period('M')).size())

2018-01-31 23:38:00
2015-01-01 00:00:00
order date (DateOrders)
2017-12    2119
2018-01    2122
Freq: M, dtype: int64


In [24]:
recent = df[df['order date (DateOrders)'] >= '2017-08-01']
print(recent.groupby(recent['order date (DateOrders)'].dt.to_period('M')).size())

order date (DateOrders)
2017-08    5305
2017-09    5189
2017-10    2253
2017-11    2052
2017-12    2119
2018-01    2122
Freq: M, dtype: int64
